# Week 51, Day 2: History: what the last trials do to the next

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w051/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
d = np.load(dataset("ibl_biased.npz"))
mouse, session = d["mouse"].astype(int), d["session"].astype(int)
contrast, choice, reward = d["contrast"].astype(float), d["choice"].astype(float), d["reward"].astype(int)
p_left, rt = d["p_left"].astype(float).round(2), d["rt"].astype(float)
ok = (choice >= 0) & d["included"] & np.isfinite(rt)
sess = 10 * mouse + session                       # a unique id for each session
c_pm = np.where(choice >= 0, 2 * choice - 1, 0.0) # +1 right, -1 left, 0 no choice
f_c = np.tanh(5 * contrast) / np.tanh(5)          # the IBL's compressed contrast regressor

def lag(x, k):
    """x shifted back by k trials, with zeros where trial t - k is in another session."""
    out = np.zeros_like(x, dtype=float); out[k:] = x[:-k]
    same = np.r_[np.zeros(k, bool), sess[k:] == sess[:-k]]
    return np.where(same, out, 0.0)

def logreg(X, y, l2=1e-3):
    """Logistic regression by Newton's method with a tiny ridge penalty."""
    w = np.zeros(X.shape[1])
    for _ in range(25):
        p = 1 / (1 + np.exp(-X @ w))
        g = X.T @ (y - p) - l2 * w
        H = -(X * (p * (1 - p))[:, None]).T @ X - l2 * np.eye(len(w))
        w -= np.linalg.solve(H, g)
    return w
print("choices on the first six trials:", c_pm[:6], "\nthe same, one trial back:", lag(c_pm, 1)[:6])

### Exercise: Win-stay, lose-what?

Build X with the four columns described above, using lag and the coded choice c_pm. Then fit logreg to each mouse's usable trials and store the weights in W.

In [ ]:
X = ...
W = np.zeros((18, 4))
for m in range(18):
    k = ok & (mouse == m)
    W[m] = ...
names = ["contrast", "rewarded prev", "unrewarded prev", "bias"]
print("mean weights:", {n: round(float(w), 2) for n, w in zip(names, W.mean(0))})
print(f"repeat after reward: {np.sum(W[:, 1] > 0)} of 18 mice; repeat after error: {np.sum(W[:, 2] > 0)} of 18")
plt.scatter(W[:, 1], W[:, 2]); plt.axhline(0, color="grey"); plt.axvline(0, color="grey")
plt.xlabel("weight of a rewarded previous choice"); plt.ylabel("weight of an unrewarded previous choice"); plt.show()

In [ ]:
import numpy as np
def test_X():
    "X has columns f_c, previous rewarded choice, previous unrewarded choice and a constant"
    exp = np.c_[f_c, lag(c_pm * reward, 1), lag(c_pm * (1 - reward), 1), np.ones(len(f_c))]
    assert X.shape == exp.shape and np.allclose(X, exp)
def test_W():
    "W holds one weight vector per mouse, fitted on that mouse's usable trials"
    k = ok & (mouse == 5)
    assert W.shape == (18, 4) and np.allclose(W[5], logreg(X[k], choice[k]), atol=1e-6)
def test_signs():
    "every mouse repeats rewarded choices, and most repeat unrewarded ones too"
    assert np.all(W[:, 1] > 0) and np.sum(W[:, 2] > 0) >= 15

test_X()
test_W()
test_signs()
print("All checks passed")

### Exercise: Held-out gain from history

Write mean_ll(X, y, w), the mean log-likelihood per trial of a logistic model (use np.logaddexp(0, z) for log(1 + e^z)). Then, for each mouse, fit the stimulus-only model (columns 0 and 3) and the full model on sessions 0 to 3, and store the difference in held-out log-likelihood on session 4 in gain.

In [ ]:
def mean_ll(X, y, w):
    z = X @ w
    return ...

gain = np.zeros(18)
for m in range(18):
    tr = ok & (mouse == m) & (session < 4)
    te = ok & (mouse == m) & (session == 4)
    base = mean_ll(X[te][:, [0, 3]], choice[te], logreg(X[tr][:, [0, 3]], choice[tr]))
    full = ...
    gain[m] = full - base
print("held-out gain per trial (nats):", np.round(gain, 3))
print(f"history helps {np.sum(gain > 0)} of 18 mice; median gain {np.median(gain):.3f} nats per trial")

In [ ]:
import numpy as np
def test_ll():
    "mean_ll is the mean Bernoulli log-likelihood per trial"
    Xt = np.c_[np.ones(4), [0, 1, 2, 3]]; yt = np.array([0, 1, 1, 0.0]); w = np.array([0.1, 0.3])
    z = Xt @ w; assert abs(mean_ll(Xt, yt, w) - np.mean(yt * z - np.log1p(np.exp(z)))) < 1e-12
def test_gain():
    "gain has one value per mouse and history helps at least 15 of them on held-out data"
    assert gain.shape == (18,) and np.sum(gain > 0) >= 15

test_ll()
test_gain()
print("All checks passed")

### Exercise: The choice history kernel

Build XL from f_c, lag(c_pm, k) for k = 1 to 10, and a constant. Fit each mouse, average the ten lag weights over mice into kernel, and estimate the decay: fit a straight line to log(kernel) against lag for lags 1 to 6, and convert its slope s into alpha = 1 - exp(s).

In [ ]:
XL = ...
WL = np.array([logreg(XL[ok & (mouse == m)], choice[ok & (mouse == m)]) for m in range(18)])
kernel = ...
slope = ...
alpha = 1 - np.exp(slope)
print("mean kernel:", np.round(kernel, 3))
print(f"each step back multiplies the weight by about {np.exp(slope):.2f}: alpha = {alpha:.2f}, a timescale of {1 / alpha:.1f} trials")
lags_ = np.arange(1, 11)
plt.plot(lags_, kernel, "o-", label="mean over mice")
plt.plot(lags_, kernel[0] * (1 - alpha) ** (lags_ - 1), "--", label="exponential")
plt.xlabel("trials back"); plt.ylabel("weight"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_XL():
    "XL has f_c, the ten lagged choices and a constant"
    assert XL.shape == (len(f_c), 12) and np.allclose(XL[:, 3], lag(c_pm, 3)) and np.allclose(XL[:, -1], 1)
def test_kernel():
    "kernel is the mean over mice of the ten lag weights, and it decays"
    assert kernel.shape == (10,) and kernel[0] > kernel[4] > 0
def test_alpha():
    "alpha comes from the slope of log weight against lag over the first six lags"
    slope = np.polyfit(np.arange(1, 7), np.log(kernel[:6]), 1)[0]
    assert abs(alpha - (1 - np.exp(slope))) < 1e-9 and 0.1 < alpha < 0.4

test_XL()
test_kernel()
test_alpha()
print("All checks passed")